# Treinamento Final — Random Forest (Dataset Brasileiro)

**Modelo:** `RandomForestClassifier` (scikit-learn)  
**Dataset:** `dataset_phishing_brasileiro.csv`  
**Objetivo:** Treinar e salvar o modelo final para detecção de phishing em URLs brasileiras

### Pipeline
1. Monta Drive e carrega CSV
2. Extrai 34 features numéricas por URL
3. **Busca de hiperparâmetros** via `RandomizedSearchCV`
4. Treina modelo final com melhores params
5. Avalia (métricas, curvas, feature importance)
6. Salva modelo `.pkl` em `/MyDrive/phishing_randomforest/`

### Referência — Benchmark (708k URLs, kmack/Phishing_urls)
| Métrica | GBM Brasileiro | Random Forest Benchmark |
|---|---|---|
| F1-Score | 87.53% | 84.2% |
| AUC-ROC | 94.68% | 91.6% |
| MCC | 0.7487 | 0.677 |
| Latência P50 | 10.23ms | 45ms |

**Antes de rodar:**
1. Faça upload do `dataset_phishing_brasileiro.csv` no Google Drive
2. Ajuste `CSV_PATH` na célula de configuração se necessário

In [ ]:
# ============================================================
# 1. Instalação de dependências
# ============================================================
!pip install -q scikit-learn pandas numpy matplotlib seaborn tldextract joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import re
import warnings
import os
warnings.filterwarnings('ignore')

from urllib.parse import urlparse
import tldextract
import joblib

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, log_loss, matthews_corrcoef,
    precision_recall_curve, average_precision_score
)

print('Imports OK.')

In [ ]:
# ============================================================
# 2. Montar Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado com sucesso.')

In [ ]:
# ============================================================
# 3. Configuração
# ============================================================
CSV_PATH     = '/content/drive/MyDrive/dataset_phishing_brasileiro.csv'
OUTPUT_DIR   = '/content/drive/MyDrive/phishing_randomforest'

RANDOM_STATE = 42
TEST_SIZE    = 0.2
CV_FOLDS     = 5
N_ITER_SEARCH = 30

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Pasta de saída: {OUTPUT_DIR}')

In [ ]:
# ============================================================
# 4. Carregar Dataset
# ============================================================
df = pd.read_csv(CSV_PATH)

print(f'Total de registros: {len(df):,}')
print(f'Colunas: {list(df.columns)}')
print(f'\nDistribuição de labels:')
print(df['label'].value_counts())
print(f'  Phishing:  {df["label"].sum():,} ({df["label"].mean():.1%})')
print(f'  Legítimo:  {(df["label"] == 0).sum():,} ({(df["label"] == 0).mean():.1%})')

if 'source' in df.columns:
    print(f'\nDistribuição por fonte:')
    print(df['source'].value_counts())

df.head(10)

In [ ]:
# ============================================================
# 5. Feature Engineering
#    34 features numéricas — idênticas ao notebook GBM
# ============================================================

def extract_features(url: str) -> dict:
    """Extrai 34 features numéricas de uma URL."""
    raw = url
    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url

    try:
        parsed = urlparse(url)
    except Exception:
        parsed = urlparse('http://invalid')

    try:
        ext = tldextract.extract(url)
    except Exception:
        ext = tldextract.extract('http://invalid')

    hostname = parsed.hostname or ''
    path     = parsed.path or ''
    query    = parsed.query or ''

    features = {}

    # Comprimento
    features['url_length']      = len(raw)
    features['hostname_length'] = len(hostname)
    features['path_length']     = len(path)
    features['query_length']    = len(query)

    # Contagens de caracteres
    features['num_dots']        = raw.count('.')
    features['num_hyphens']     = raw.count('-')
    features['num_underscores'] = raw.count('_')
    features['num_slashes']     = raw.count('/')
    features['num_ats']         = raw.count('@')
    features['num_ampersands']  = raw.count('&')
    features['num_equals']      = raw.count('=')
    features['num_digits']      = sum(c.isdigit() for c in raw)
    features['num_special']     = sum(not c.isalnum() and c not in './-_' for c in raw)
    features['num_params']      = query.count('=') if query else 0

    # Proporções
    features['digit_ratio']   = features['num_digits']  / max(len(raw), 1)
    features['special_ratio'] = features['num_special'] / max(len(raw), 1)

    # Profundidade do path
    features['path_depth'] = path.count('/') - 1 if path else 0

    # Domínio
    features['subdomain_length'] = len(ext.subdomain)
    features['domain_length']    = len(ext.domain)
    features['tld_length']       = len(ext.suffix)
    features['num_subdomains']   = ext.subdomain.count('.') + 1 if ext.subdomain else 0

    # Padrões suspeitos
    features['has_ip'] = int(bool(re.match(
        r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', hostname
    )))
    features['has_at_sign']               = int('@' in raw)
    features['has_double_slash_redirect'] = int('//' in raw[8:])
    features['has_hex_encoding']          = int('%' in raw)
    features['has_https']                 = int(raw.startswith('https'))
    features['has_www']                   = int('www.' in raw.lower())
    try:
        port = parsed.port
    except ValueError:
        port = None
    features['has_port'] = int(port is not None and port not in [80, 443])

    # Entropia de Shannon
    if len(raw) > 0:
        prob = [raw.count(c) / len(raw) for c in set(raw)]
        features['entropy'] = -sum(p * np.log2(p) for p in prob if p > 0)
    else:
        features['entropy'] = 0

    # Extensão suspeita — .html/.htm excluídos (falso positivo em sites gov/.br)
    suspicious_ext = ['.exe', '.zip', '.rar', '.js', '.php',
                      '.cgi', '.asp', '.aspx', '.scr', '.bat', '.cmd']
    features['has_suspicious_ext'] = int(any(path.lower().endswith(e) for e in suspicious_ext))

    # Palavras-chave de phishing
    phishing_keywords = ['login', 'signin', 'verify', 'account', 'update',
                         'secure', 'banking', 'confirm', 'password', 'suspend',
                         'alert', 'paypal', 'ebay', 'amazon', 'apple', 'microsoft']
    raw_lower = raw.lower()
    features['num_phishing_keywords'] = sum(kw in raw_lower for kw in phishing_keywords)

    # Tokens
    tokens = re.split(r'[.\-_/=?&]', raw)
    features['num_tokens']       = len(tokens)
    features['avg_token_length'] = np.mean([len(t) for t in tokens if t]) if tokens else 0
    features['max_token_length'] = max((len(t) for t in tokens if t), default=0)

    return features


print('Extraindo features das URLs...')
t0 = time.time()

url_col = 'url' if 'url' in df.columns else 'text'
feature_rows = df[url_col].apply(extract_features)
df_features  = pd.DataFrame(feature_rows.tolist())

print(f'Features extraídas em {time.time() - t0:.1f}s')
print(f'Shape: {df_features.shape} — {df_features.shape[1]} features')
df_features.head()

In [ ]:
# ============================================================
# 6. Pré-processamento e Split
# ============================================================
X = df_features.copy()
y = df['label'].astype(int)

X = X.replace([np.inf, -np.inf], np.nan)
nulos = X.isnull().sum().sum()
if nulos > 0:
    print(f'Valores nulos: {nulos} — preenchendo com mediana.')
    X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,} | Features: {X_train.shape[1]}')
print(f'Train — Phishing: {y_train.sum():,} ({y_train.mean():.1%}) | Legítimo: {(~y_train.astype(bool)).sum():,}')
print(f'Test  — Phishing: {y_test.sum():,}  ({y_test.mean():.1%}) | Legítimo: {(~y_test.astype(bool)).sum():,}')

In [ ]:
# ============================================================
# 7. Busca de Hiperparâmetros (RandomizedSearchCV)
#
# Por que cada parâmetro importa:
#   n_estimators    — número de árvores; mais = melhor mas mais lento
#   max_depth       — profundidade máxima; None = cresce até folhas puras
#   min_samples_leaf — mínimo de amostras por folha (regularização)
#   min_samples_split — mínimo para dividir um nó
#   max_features    — features consideradas por split ('sqrt' = padrão RF)
#   class_weight    — balanceamento de classes
# ============================================================

param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_leaf':  [1, 2, 5, 10],
    'min_samples_split': [2, 5, 10],
    'max_features':      ['sqrt', 'log2', 0.3, 0.5],
}

base_estimator = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

cv_search = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=base_estimator,
    param_distributions=param_dist,
    n_iter=N_ITER_SEARCH,
    scoring='f1',
    cv=cv_search,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE,
    refit=False,
)

print(f'Iniciando RandomizedSearchCV ({N_ITER_SEARCH} combinações × 3 folds)...')
t0 = time.time()
search.fit(X_train, y_train)
search_time = time.time() - t0

best_params = search.best_params_
print(f'\nConcluído em {search_time:.1f}s')
print(f'Melhor F1 (CV 3-fold): {search.best_score_:.4f}')
print(f'Melhores parâmetros encontrados:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

# ------------------------------------------------------------
# Após rodar uma vez, cole os best_params aqui para pular a busca:
# ------------------------------------------------------------
# best_params = {
#     'n_estimators':      300,
#     'max_depth':         None,
#     'min_samples_leaf':  2,
#     'min_samples_split': 2,
#     'max_features':      'sqrt',
# }

In [ ]:
# ============================================================
# 8. Treino do Modelo Final
# ============================================================
print('Treinando Random Forest com melhores hiperparâmetros...')
print(f'Params: {best_params}')

rf = RandomForestClassifier(
    **best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

t_start = time.time()
rf.fit(X_train, y_train)
train_time = time.time() - t_start

print(f'\nTreino concluído em {train_time:.2f}s')
print(f'Árvores treinadas: {rf.n_estimators}')

In [ ]:
# ============================================================
# 9. Avaliação no Conjunto de Teste
# ============================================================
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

acc         = accuracy_score(y_test, y_pred)
prec        = precision_score(y_test, y_pred)
rec         = recall_score(y_test, y_pred)
f1          = f1_score(y_test, y_pred)
auc_roc     = roc_auc_score(y_test, y_proba)
ap          = average_precision_score(y_test, y_proba)
mcc         = matthews_corrcoef(y_test, y_pred)
logloss     = log_loss(y_test, y_proba)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
specificity = tn / (tn + fp)

latencies = []
for i in range(min(500, len(X_test))):
    sample = X_test.iloc[[i]]
    t0 = time.perf_counter()
    rf.predict(sample)
    latencies.append((time.perf_counter() - t0) * 1000)
latencies = np.array(latencies)

# Referência — GBM Brasileiro
BENCH_F1      = 0.8753
BENCH_AUC_ROC = 0.9468
BENCH_MCC     = 0.7487

print('=' * 65)
print('RESULTADOS — Random Forest Final (Dataset Brasileiro)')
print('=' * 65)
print(f'{"Métrica":<20} {"Atual":>10}   {"GBM Brasileiro":>14}   {"Delta":>8}')
print('-' * 65)
print(f'{"Accuracy":<20} {acc:>10.4f}')
print(f'{"Precision":<20} {prec:>10.4f}')
print(f'{"Recall":<20} {rec:>10.4f}')
print(f'{"F1-Score":<20} {f1:>10.4f}   {BENCH_F1:>14.4f}   {f1 - BENCH_F1:>+8.4f}')
print(f'{"AUC-ROC":<20} {auc_roc:>10.4f}   {BENCH_AUC_ROC:>14.4f}   {auc_roc - BENCH_AUC_ROC:>+8.4f}')
print(f'{"MCC":<20} {mcc:>10.4f}   {BENCH_MCC:>14.4f}   {mcc - BENCH_MCC:>+8.4f}')
print(f'{"PR-AUC":<20} {ap:>10.4f}')
print(f'{"Log Loss":<20} {logloss:>10.4f}')
print(f'{"Specificity":<20} {specificity:>10.4f}')
print(f'{"Latência P50 (ms)":<20} {np.percentile(latencies, 50):>10.2f}')
print(f'{"Latência P95 (ms)":<20} {np.percentile(latencies, 95):>10.2f}')
print(f'{"Latência P99 (ms)":<20} {np.percentile(latencies, 99):>10.2f}')
print(f'{"Árvores":<20} {rf.n_estimators:>10}')
print(f'{"Tempo treino (s)":<20} {train_time:>10.2f}')
print('=' * 65)
print()
print(classification_report(y_test, y_pred, target_names=['Legítimo', 'Phishing']))

In [ ]:
# ============================================================
# 10. Cross-Validation (5-Fold)
# ============================================================
print(f'Executando Cross-Validation ({CV_FOLDS}-Fold)...')

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_scores = cross_validate(
    RandomForestClassifier(**best_params, random_state=RANDOM_STATE, n_jobs=-1),
    X, y,
    cv=cv,
    scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc', 'precision': 'precision', 'recall': 'recall'},
    n_jobs=-1
)

cv_f1_mean  = cv_scores['test_f1'].mean()
cv_f1_std   = cv_scores['test_f1'].std()
cv_auc_mean = cv_scores['test_roc_auc'].mean()
cv_auc_std  = cv_scores['test_roc_auc'].std()

print('Cross-Validation Results:')
print(f'  {"Métrica":<12} {"mean±std":>22}')
print('  ' + '-' * 35)
print(f'  {"accuracy":<12} {cv_scores["test_accuracy"].mean():.4f} ± {cv_scores["test_accuracy"].std():.4f}')
print(f'  {"f1":<12} {cv_f1_mean:.4f} ± {cv_f1_std:.4f}')
print(f'  {"roc_auc":<12} {cv_auc_mean:.4f} ± {cv_auc_std:.4f}')
print(f'  {"precision":<12} {cv_scores["test_precision"].mean():.4f} ± {cv_scores["test_precision"].std():.4f}')
print(f'  {"recall":<12} {cv_scores["test_recall"].mean():.4f} ± {cv_scores["test_recall"].std():.4f}')
print(f'\n  F1 por fold: {[f"{v:.4f}" for v in cv_scores["test_f1"]]}')

if cv_f1_std < 0.01:
    print(f'\n  Modelo estável (std F1 = {cv_f1_std:.4f} < 0.01)')
else:
    print(f'\n  ATENÇÃO: variância alta entre folds (std = {cv_f1_std:.4f})')

In [ ]:
# ============================================================
# 11. Visualizações
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Legítimo', 'Phishing'])
disp.plot(ax=ax, cmap='Greens', colorbar=False, values_format='d')
ax.set_title('Matriz de Confusão', fontsize=13, fontweight='bold')

ax = axes[1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_val = auc(fpr, tpr)
ax.plot(fpr, tpr, color='forestgreen', lw=2, label=f'AUC = {roc_val:.4f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.50)')
ax.fill_between(fpr, tpr, alpha=0.1, color='forestgreen')
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('Curva ROC', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

ax = axes[2]
prec_vals, rec_vals, _ = precision_recall_curve(y_test, y_proba)
ax.plot(rec_vals, prec_vals, color='darkolivegreen', lw=2, label=f'AP = {ap:.4f}')
ax.fill_between(rec_vals, prec_vals, alpha=0.1, color='darkolivegreen')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curva Precision-Recall', fontsize=13, fontweight='bold')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)

plt.suptitle('Random Forest — Dataset Brasileiro', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'curvas_avaliacao.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvo: {plot_path}')

In [ ]:
# ============================================================
# 12. Importância das Features
# ============================================================
importances = rf.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
feat_imp.tail(20).plot(kind='barh', ax=ax, color='forestgreen', edgecolor='black', linewidth=0.5)
ax.set_title('Top 20 Features — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importância')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()

fi_path = os.path.join(OUTPUT_DIR, 'feature_importance.png')
plt.savefig(fi_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvo: {fi_path}')

df_fi = feat_imp.sort_values(ascending=False).reset_index()
df_fi.columns = ['Feature', 'Importância']
print('\nTop 10 features mais importantes:')
print(df_fi.head(10).to_string(index=False))

In [ ]:
# ============================================================
# 13. Salvar Modelo e Resultados no Drive
# ============================================================
model_path = os.path.join(OUTPUT_DIR, 'rf_phishing_brasileiro.pkl')
joblib.dump(rf, model_path)
print(f'Modelo salvo: {model_path}')

feature_names = list(X.columns)
features_path = os.path.join(OUTPUT_DIR, 'feature_names.pkl')
joblib.dump(feature_names, features_path)
print(f'Feature names salvo: {features_path}')

results_dict = {
    'Modelo':            ['Random Forest'],
    'Dataset':           ['dataset_phishing_brasileiro'],
    'Accuracy':          [round(acc, 4)],
    'Precision':         [round(prec, 4)],
    'Recall':            [round(rec, 4)],
    'F1-Score':          [round(f1, 4)],
    'AUC-ROC':           [round(auc_roc, 4)],
    'PR-AUC':            [round(ap, 4)],
    'MCC':               [round(mcc, 4)],
    'Log Loss':          [round(logloss, 4)],
    'Specificity':       [round(specificity, 4)],
    'CV_F1_mean':        [round(cv_f1_mean, 4)],
    'CV_F1_std':         [round(cv_f1_std, 4)],
    'CV_AUC_mean':       [round(cv_auc_mean, 4)],
    'CV_AUC_std':        [round(cv_auc_std, 4)],
    'Latencia_P50_ms':   [round(np.percentile(latencies, 50), 2)],
    'Latencia_P95_ms':   [round(np.percentile(latencies, 95), 2)],
    'Latencia_P99_ms':   [round(np.percentile(latencies, 99), 2)],
    'n_estimators':      [rf.n_estimators],
    'Tempo_Treino_s':    [round(train_time, 2)],
    'N_train':           [len(X_train)],
    'N_test':            [len(X_test)],
    'best_params':       [str(best_params)],
}
df_results = pd.DataFrame(results_dict)
results_path = os.path.join(OUTPUT_DIR, 'resultados_rf_brasileiro.csv')
df_results.to_csv(results_path, index=False)
print(f'Resultados salvos: {results_path}')

print('\n' + '='*65)
print('Arquivos salvos em:', OUTPUT_DIR)
print('  rf_phishing_brasileiro.pkl         — modelo treinado')
print('  feature_names.pkl                  — lista de features (34)')
print('  resultados_rf_brasileiro.csv       — métricas completas')
print('  curvas_avaliacao.png               — ROC / PR / confusão')
print('  feature_importance.png             — importância das features')
print('='*65)

In [ ]:
# ============================================================
# 14. Converter para ONNX (para uso na extensão Chrome)
# ============================================================
!pip install -q skl2onnx

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

onnx_model = convert_sklearn(
    rf,
    initial_types=[('float_input', FloatTensorType([None, rf.n_features_in_]))],
    options={'zipmap': False}   # saída Float32Array em vez de ZipMap
)
onnx_model.ir_version = 8      # ORT Web 1.14 suporta até IR version 8

onnx_path = os.path.join(OUTPUT_DIR, 'rf_phishing_brasileiro.onnx')
with open(onnx_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())

print(f'ONNX salvo: {onnx_path}')
print(f'IR version: {onnx_model.ir_version}')
print(f'Opset: {onnx_model.opset_import[0].version}')
print(f'Features: {rf.n_features_in_}')

In [ ]:
# ============================================================
# 15. Exemplo de Inferência com o Modelo Salvo
# ============================================================
modelo_carregado    = joblib.load(model_path)
features_carregadas = joblib.load(features_path)

def predict_url(url: str) -> dict:
    feats   = extract_features(url)
    X_input = pd.DataFrame([feats])[features_carregadas]
    X_input = X_input.replace([np.inf, -np.inf], np.nan).fillna(0)
    pred    = modelo_carregado.predict(X_input)[0]
    proba   = modelo_carregado.predict_proba(X_input)[0][1]
    return {
        'url': url,
        'predicao': 'PHISHING' if pred == 1 else 'LEGITIMO',
        'confianca_phishing': f'{proba:.2%}'
    }

urls_teste = [
    'https://presidencia.gov.br',
    'https://www.planalto.gov.br/ccivil_03/constituicao/constituicao.htm',
    'https://www.leagueoflegends.com/pt-br/',
    'http://192.168.1.1/login?account=verify&password=update',
    'https://bradesco.com.br',
    'http://bradesco-conta-bloqueada.xyz/login/verify/account',
    'https://itau.com.br/conta/acesso',
    'http://itau-seguranca-conta.ml/login?token=abc123',
]

print('Exemplos de predição:')
print('-' * 75)
for url in urls_teste:
    r = predict_url(url)
    print(f"  [{r['predicao']:8s}] {r['confianca_phishing']:6s}  {r['url']}")